# Schmucker Art Gallery - Virtual Exhibition Tour Builder

Upload your own artwork images and fill in the details below. The notebook builds a polished interactive HTML page you can open in any browser.

**Steps:**
1. Run Cell 1
2. Edit Cell 2 with your artwork info
3. Run Cell 3 to upload your images
4. Run Cell 4 to build and download the HTML

In [ ]:
# CELL 1 - Ready check
print('Ready! Proceed to Cell 2.')


In [ ]:
# CELL 2 - Fill in your artwork info here
# Set image_file to the filename you will upload in Cell 3

EXHIBITION_TITLE    = 'Making Her Mark'
EXHIBITION_SUBTITLE = 'Fifty Women Artists of the Historic Woodstock Art Colony'
GALLERY_NAME        = 'Schmucker Art Gallery, Gettysburg College'
EXHIBITION_DATES    = 'January 28 - April 11, 2026'
INTRO_TEXT = (
    'This exhibition celebrates the remarkable women who shaped one of '
    "America's most storied artist colonies. Working in Woodstock, New York "
    'from the early twentieth century onward, these painters, printmakers, '
    'and sculptors forged careers outside the mainstream art world.'
)

ARTWORKS = [
    {
        'title':      'Landscape at Dusk',
        'artist':     'Jane Smith',
        'date':       '1923',
        'medium':     'Oil on canvas',
        'dimensions': '24 x 36 in.',
        'credit':     "Gift of the artist's estate",
        'bio':        'Jane Smith (1890-1950) was a leading figure of the Woodstock Art Colony.',
        'image_file': 'landscape.jpg',
    },
    {
        'title':      'Summer Garden',
        'artist':     'Mary Johnson',
        'date':       '1931',
        'medium':     'Watercolor on paper',
        'dimensions': '18 x 24 in.',
        'credit':     'Museum purchase',
        'bio':        'Mary Johnson studied under Robert Henri before settling in Woodstock.',
        'image_file': 'garden.jpg',
    },
    # Add more artworks here...
]

print(f'Defined {len(ARTWORKS)} artworks.')
print('Titles:', [a['title'] for a in ARTWORKS])


In [ ]:
# CELL 3 - Upload your image files
import base64, io
from IPython.display import display, HTML
from google.colab import output
from PIL import Image

image_data = {}

def process_upload(filename, b64_data):
    raw = base64.b64decode(b64_data)
    img = Image.open(io.BytesIO(raw)).convert('RGB')
    w, h = img.size
    if max(w, h) > 1200:
        scale = 1200 / max(w, h)
        img = img.resize((int(w*scale), int(h*scale)), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=82, optimize=True)
    b64_out = base64.b64encode(buf.getvalue()).decode('utf-8')
    image_data[filename] = f'data:image/jpeg;base64,{b64_out}'
    print(f'Loaded: {filename} -> {img.size[0]}x{img.size[1]}, {len(buf.getvalue()):,} bytes')

output.register_callback('notebook.process_upload', process_upload)

display(HTML('''
<input type="file" id="uploader" multiple accept="image/*">
<p id="status">Select your image files above, then wait for confirmation below.</p>
<script>
document.getElementById('uploader').addEventListener('change', function(e) {
  const files = Array.from(e.target.files);
  document.getElementById('status').innerText = 'Uploading ' + files.length + ' file(s)...';
  let done = 0;
  files.forEach(file => {
    const reader = new FileReader();
    reader.onload = function(ev) {
      const b64 = ev.target.result.split(',')[1];
      google.colab.kernel.invokeFunction('notebook.process_upload', [file.name, b64], {});
      done++;
      if (done === files.length) {
        document.getElementById('status').innerText = 'All ' + done + ' file(s) uploaded!';
      }
    };
    reader.readAsDataURL(file);
  });
});
</script>
'''))
# After all files uploaded, match to artworks
# Run this cell again or scroll down to check matches
for a in ARTWORKS:
    fname = a.get('image_file', '')
    a['_img_src'] = image_data.get(fname, '')
    status = 'OK' if a['_img_src'] else ('no file set' if not fname else f'NOT FOUND: {fname}')
    print(f"  {a['title']}: {status}")


In [ ]:
# CELL 4 - Build and download the HTML page

SITE_URL = 'https://lehafeli02.github.io/schmucker-exhibition'

CSS = (
    ':root{--cream:#f5f0e8;--warm:#faf8f4;--dark:#1c1a17;--mid:#6b6358;--lt:#a89e93;--acc:#8b4a2f;--accl:#c47a5a;--bdr:#ddd8cf}\n'
    '*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}\n'
    "body{font-family:'DM Sans',sans-serif;background:var(--cream);color:var(--dark)}\n"
    '.hero{background:var(--dark);color:var(--cream);padding:72px 48px 60px;position:relative;overflow:hidden}\n'
    ".hero::before{content:'';position:absolute;inset:0;background:radial-gradient(ellipse at 70% 50%,rgba(139,74,47,.25),transparent 65%)}\n"
    '.hi{max-width:800px;position:relative}\n'
    '.gn{font-size:11px;font-weight:500;letter-spacing:.18em;text-transform:uppercase;color:var(--accl);margin-bottom:22px}\n'
    ".hero h1{font-family:'Playfair Display',Georgia,serif;font-size:clamp(2.2rem,5vw,3.8rem);font-weight:700;line-height:1.1;margin-bottom:12px}\n"
    ".hs{font-family:'Playfair Display',serif;font-style:italic;font-size:clamp(.95rem,2vw,1.25rem);color:var(--lt);margin-bottom:28px;max-width:580px}\n"
    '.hd{display:inline-block;border:1px solid rgba(255,255,255,.2);padding:5px 14px;font-size:11px;letter-spacing:.1em;text-transform:uppercase;color:var(--lt);margin-bottom:36px}\n'
    '.hint{font-size:1rem;line-height:1.7;color:rgba(245,240,232,.8);max-width:620px;font-weight:300}\n'
    '.ctrl{background:var(--warm);border-bottom:1px solid var(--bdr);padding:16px 48px;position:sticky;top:0;z-index:50;display:flex;gap:16px;align-items:center}\n'
    '.sw{flex:1;position:relative;max-width:420px}\n'
    '#si{width:100%;padding:9px 14px 9px 38px;border:1px solid var(--bdr);border-radius:4px;font-size:14px;background:var(--cream);color:var(--dark);outline:none;transition:border-color .2s;font-family:inherit}\n'
    '#si:focus{border-color:var(--acc)}\n'
    '#si::placeholder{color:var(--lt)}\n'
    '.sico{position:absolute;left:12px;top:50%;transform:translateY(-50%);color:var(--lt);font-size:14px;pointer-events:none}\n'
    '.cnt{font-size:13px;color:var(--mid);white-space:nowrap}\n'
    '.gs{padding:44px 48px}\n'
    '.sl{font-size:11px;font-weight:500;letter-spacing:.14em;text-transform:uppercase;color:var(--mid);margin-bottom:30px;padding-bottom:11px;border-bottom:1px solid var(--bdr)}\n'
    '.grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(290px,1fr));gap:2px}\n'
    '.card{background:var(--warm);cursor:pointer;transition:transform .25s,box-shadow .25s;animation:fu .5s ease both}\n'
    '@keyframes fu{from{opacity:0;transform:translateY(16px)}to{opacity:1;transform:translateY(0)}}\n'
    '.card:hover{transform:translateY(-3px);box-shadow:0 10px 28px rgba(28,26,23,.11);z-index:2}\n'
    '.cimg{width:100%;aspect-ratio:4/3;overflow:hidden;background:var(--cream)}\n'
    '.cimg img{width:100%;height:100%;object-fit:cover;display:block;transition:transform .4s}\n'
    '.card:hover .cimg img{transform:scale(1.04)}\n'
    '.img-ph{width:100%;height:100%;display:flex;align-items:center;justify-content:center;background:linear-gradient(135deg,#ebe6dc,#d6cfc4);color:var(--lt);font-size:11px;letter-spacing:.08em;text-transform:uppercase}\n'
    '.cbody{padding:18px 20px 22px}\n'
    '.ca{font-size:10.5px;font-weight:500;letter-spacing:.1em;text-transform:uppercase;color:var(--acc);margin-bottom:5px}\n'
    ".ct{font-family:'Playfair Display',serif;font-size:1.1rem;font-weight:400;line-height:1.3;margin-bottom:7px}\n"
    '.cd{font-size:12px;color:var(--mid);margin-bottom:3px}\n'
    '.cm{font-size:11.5px;color:var(--lt);font-style:italic;margin-bottom:11px;line-height:1.4}\n'
    '.bio{font-size:12.5px;color:var(--mid);line-height:1.6;margin-bottom:9px;padding-top:11px;border-top:1px solid var(--bdr)}\n'
    '.cc{font-size:11px;color:var(--lt);line-height:1.5}\n'
    '.empty{display:none;text-align:center;padding:80px 48px;color:var(--mid)}\n'
    '.empty.on{display:block}\n'
    'footer{text-align:center;padding:36px 48px;border-top:1px solid var(--bdr);font-size:11.5px;color:var(--lt);letter-spacing:.04em}\n'
    '@media(max-width:640px){.hero,.gs{padding:28px 20px}.ctrl{padding:12px 20px}.grid{grid-template-columns:1fr}}\n'
)

JS = (
    "const cards=document.querySelectorAll('.card'),inp=document.getElementById('si'),"
    "cnt=document.getElementById('cnt'),empty=document.getElementById('empty');\n"
    "inp.addEventListener('input',()=>{"
    "const q=inp.value.toLowerCase();let v=0;"
    "cards.forEach(c=>{const s=!q||c.textContent.toLowerCase().includes(q);"
    "c.style.display=s?'':'none';if(s)v++;});"
    "cnt.textContent=v+' work'+(v!==1?'s':'');"
    "empty.classList.toggle('on',v===0);});\n"
)

SHARE_HTML = (
    '<div id="sb" style="position:fixed;bottom:24px;right:24px;z-index:100;display:flex;flex-direction:column;align-items:flex-end;gap:10px">'
    '<div id="qb" style="display:none;background:#fff;border:1px solid #ddd8cf;border-radius:8px;padding:16px;box-shadow:0 8px 24px rgba(0,0,0,.12);text-align:center">'
    '<p style="font-size:11px;letter-spacing:.08em;text-transform:uppercase;color:#6b6358;margin-bottom:10px">Scan to visit</p>'
    '<div id="qi"></div>'
    f'<p style="font-size:10px;color:#a89e93;margin-top:8px;max-width:140px;word-break:break-all">{SITE_URL}</p>'
    '</div>'
    '<div style="display:flex;gap:8px">'
    '<button onclick="toggleQR()" style="background:#1c1a17;color:#f5f0e8;border:none;padding:10px 16px;border-radius:4px;font-size:12px;letter-spacing:.06em;cursor:pointer;font-family:inherit">QR Code</button>'
    '<button onclick="sharePage()" style="background:#8b4a2f;color:#fff;border:none;padding:10px 16px;border-radius:4px;font-size:12px;letter-spacing:.06em;cursor:pointer;font-family:inherit">Share</button>'
    '</div></div>'
    '<script src="https://cdnjs.cloudflare.com/ajax/libs/qrcodejs/1.0.0/qrcode.min.js"></script>'
    '<script>'
    'let qrMade=false;'
    'function toggleQR(){const box=document.getElementById("qb");const v=box.style.display!=="none";box.style.display=v?"none":"block";'
    'if(!qrMade){new QRCode(document.getElementById("qi"),{text:"https://lehafeli02.github.io/schmucker-exhibition",width:140,height:140,colorDark:"#1c1a17",colorLight:"#ffffff"});qrMade=true;}}'
    'function sharePage(){'
    'if(navigator.share){navigator.share({title:"Making Her Mark",url:"https://lehafeli02.github.io/schmucker-exhibition"});}'
    'else{navigator.clipboard.writeText("https://lehafeli02.github.io/schmucker-exhibition").then(()=>alert("Link copied!"));}}'
    '</script>'
)

def make_card(a, idx):
    delay = idx * 0.07
    img_src = a.get('_img_src', '')
    img_tag = (f'<img src="{img_src}" alt="{a["title"]}" loading="lazy">'
               if img_src else
               '<div class="img-ph"><span>Image Coming Soon</span></div>')
    bio_html    = f'<p class="bio">{a["bio"]}</p>' if a.get('bio') else ''
    credit_html = f'<p class="cc">{a["credit"]}</p>' if a.get('credit') else ''
    med = ' - '.join(filter(None, [a.get('medium',''), a.get('dimensions','')]))
    return (
        f'<article class="card" style="animation-delay:{delay:.2f}s">'
        f'<div class="cimg">{img_tag}</div>'
        f'<div class="cbody">'
        f'<p class="ca">{a["artist"]}</p>'
        f'<h2 class="ct">{a["title"]}</h2>'
        f'<p class="cd">{a.get("date","")}</p>'
        f'<p class="cm">{med}</p>'
        f'{bio_html}{credit_html}'
        f'</div></article>'
    )

cards_html = ''.join(make_card(a, i) for i, a in enumerate(ARTWORKS))
count = len(ARTWORKS)
plural = 's' if count != 1 else ''

html = f'''<!DOCTYPE html>
<html lang=\"en\"><head>
<meta charset=\"UTF-8\"><meta name=\"viewport\" content=\"width=device-width,initial-scale=1.0\">
<title>{EXHIBITION_TITLE} - {GALLERY_NAME}</title>
<link rel=\"preconnect\" href=\"https://fonts.googleapis.com\">
<link href=\"https://fonts.googleapis.com/css2?family=Playfair+Display:ital,wght@0,400;0,700;1,400&family=DM+Sans:wght@300;400;500&display=swap\" rel=\"stylesheet\">
<style>{CSS}</style></head><body>
<header class=\"hero\"><div class=\"hi\">
<p class=\"gn\">{GALLERY_NAME}</p>
<h1>{EXHIBITION_TITLE}</h1>
<p class=\"hs\">{EXHIBITION_SUBTITLE}</p>
<div class=\"hd\">{EXHIBITION_DATES}</div>
<p class=\"hint\">{INTRO_TEXT}</p>
</div></header>
<nav class=\"ctrl\">
<div class=\"sw\"><span class=\"sico\">&#128269;</span>
<input type=\"text\" id=\"si\" placeholder=\"Search by artist, title, or medium...\"></div>
<span class=\"cnt\" id=\"cnt\">{count} work{plural}</span>
</nav>
<main class=\"gs\"><p class=\"sl\">Works in the Exhibition</p>
<div class=\"grid\" id=\"grid\">{cards_html}</div>
<div class=\"empty\" id=\"empty\">
<p style=\"font-size:2.5rem;margin-bottom:16px\">&#128444;&#65039;</p>
<h3 style=\"font-family:'Playfair Display',serif;font-size:1.3rem;margin-bottom:8px;color:#1c1a17\">No works found</h3>
<p>Try a different search term.</p>
</div></main>
<footer>{GALLERY_NAME} | {EXHIBITION_DATES} | Free and open to the public</footer>
<script>{JS}</script>{SHARE_HTML}
</body></html>'''

OUTPUT = 'exhibition_tour.html'
with open(OUTPUT, 'w', encoding='utf-8') as f:
    f.write(html)
print(f'Built: {OUTPUT} ({len(html):,} chars, {count} artworks)')

from google.colab import files
files.download(OUTPUT)
print('Download started!')


---
## Done!

Open `exhibition_tour.html` in any browser. Images are embedded directly so the file works offline.

**To update with real Schmucker data:** edit ARTWORKS in Cell 2, re-upload images in Cell 3, re-run Cell 4.